## Selenium이용한 동적 크롤링 실습 : 주유소 가격 수집

In [52]:
from bs4 import BeautifulSoup
from urllib.request import urlopen
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

In [5]:
url = 'https://www.opinet.co.kr/searRgSelect.do'
html = urlopen(url).read()
soup = BeautifulSoup(html, 'html.parser')

In [6]:
print(html)

b'\n\n\n<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml" lang="ko" xml:lang="ko">\n<head>\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n<link rel="stylesheet" type="text/css" href="/css/sample.css"/>\n<title>error.jsp</title>\n</head>\n<body>\n<table width="100%" height="100%" border="0" cellpadding="0" cellspacing="0">\n  <tr align="center" valign="middle">\n     <td width="100%" height="100%" align="center" valign="middle" style="padding-top:150px;"> \n     <table border="0" cellspacing="0" cellpadding="0" align="center" valign="middle"> <font size=10>The service is not available.</font>\n  \n\n\t</table> </td>\n  </tr>\n</table>\n\n</body>\n</html>'


In [7]:
# selenium 4
from selenium import webdriver
from selenium.webdriver.common.by import By

## 싼 주유소 찾기 Opinet

- https://www.opinet.co.kr/user/main/mainView.do

### 서울시 구별 주유소 가격 정도 얻어오기

- 사이트에서 [싼주유소찾기] > [지역별] 메뉴 선택
- 지역은 '서울' > 원하는 구 지정
    - '시/군/구'를 선택하는 select option의 XPath 찾기

- '조회'버튼 클릭
- 조회된 주유소 정보 '엑셀저장' 버튼 클릭하여 엑셀파일 다운로드

#### 크롬 드라이버 객체 생성

In [11]:
driver = webdriver.Chrome()

#### 웹페이지 로드

In [12]:
url = 'https://www.opinet.co.kr/user/main/mainView.do'
driver.get(url)
driver.implicitly_wait(3)

#### [싼주유소찾기]-[지역] 페이지로 이동

- 자바스트립트 코드 실행으로 통해서 이동

In [13]:
# <a href="javascript:goSubPage(0,0,99);"><span>지역별</span></a>
driver.execute_script('goSubPage(0,0,99);')

#### 구 이름 찾기

In [14]:
xpath = '//*[@id="SIDO_NM0"]'
sidoList = driver.find_element(By.XPATH, xpath)
sidoOpt = sidoList.find_elements(By.TAG_NAME, 'option')
sidoNames = [opt.get_attribute('value') for opt in sidoOpt]
sidoNames

['',
 '서울특별시',
 '부산광역시',
 '대구광역시',
 '인천광역시',
 '전남광주통합특별시',
 '대전광역시',
 '울산광역시',
 '세종특별자치시',
 '경기도',
 '강원특별자치도',
 '충청북도',
 '충청남도',
 '전북특별자치도',
 '경상북도',
 '경상남도',
 '제주특별자치도']

In [15]:
len(sidoNames)

17

- 서울특별시 선택

In [17]:
sido_elem = driver.find_element(By.ID, 'SIDO_NM0')
sido_elem.send_keys('서울특별시')

#### '강남구' 주유소 지정

In [58]:
# <select style="width:108px;" id="SIGUNGU_NM0" name="SIGUNGU_NM0">
sigunguList = driver.find_element(By.CSS_SELECTOR, '#SIGUNGU_NM0')
# xpath = '//*[@id="SIGUNGU_NM0"]'
# sigunguList = driver.find_element(By.XPATH, xpath)

sigunguOpt = sigunguList.find_elements(By.TAG_NAME, 'option')
sigunguNames = [ opt.get_attribute('value') for opt in sigunguOpt ]

sigunguList.send_keys('강남구')
len(sigunguNames)

26

#### '조회' 버튼 클릭

In [35]:
xpath = '//*[@id="searRgSelect"]'
driver.find_element(By.XPATH, xpath).click()

#### 조회된 주유소 '엑셀저장' 버튼 클릭

In [36]:
xpath = '//*[@id="templ_list0"]/div[8]/div/a'
driver.find_element(By.XPATH, xpath).click()

=> '지역_위치별(주유소).xls'파일 다운로드 폴더에 저장됨

### 25개 구 주유소 엑셀파일 다운로드

- 함수로 작성하여 실행

In [59]:
# 시도 -> 구군 선택
# 구(군) 이름 지정하면 [조회]버튼 클릭 -> [엑셀저장]버튼 클릭 -> 다운로드
def search_download_GuGun(gugun:str):
    sigunguList = driver.find_element(By.CSS_SELECTOR, '#SIGUNGU_NM0')
    sigunguList.send_keys(gugun)
    time.sleep(3)

    # 조회버튼 클릭
    xpath = '//*[@id="searRgSelect"]'
    driver.find_element(By.XPATH, xpath).click()
    time.sleep(3)
    
    # 엑셀저장 버튼 클릭
    # xpath = '//*[@id="templ_list0"]/div[8]/div/a'
    # driver.find_element(By.XPATH, xpath).click()
    # time.sleep(3)
    wait = WebDriverWait(driver, 10)
    excel_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="templ_list0"]/div[8]/div/a')))
    excel_btn.click()
    time.sleep(3)

In [ ]:
# 시(도)를 선택하여 해당 시(도)의 구(군) 전체 주유소 정보 엑셀 파일 다운로드

In [49]:
def search_download_Sido(sido:str):
    #시(도) 선택
    sido_elem = driver.find_element(By.ID, 'SIDO_NM0')
    sido_elem.send_keys(sido)
    #구(군) 이름 목록 찾기
    sigunguList = driver.find_element(By.CSS_SELECTOR, '#SIGUNGU_NM0')
    
    sigunguOpt = sigunguList.find_elements(By.TAG_NAME, 'option')
    sigunguNames = [ opt.get_attribute('value') for opt in sigunguOpt ][1:]
    
    # 선택한 시도의 모든 구군 휘발유 가격 조회 및 엑셀저장 버튼 클릭
    for gugun in sigunguNames:
        search_download_GuGun(gugun)
        time.sleep(3)

In [45]:
gungu = '마포구'
search_download_GuGun(gungu)

In [54]:
sido = '서울특별시'
search_download_Sido(sido)

In [55]:
### 명시적 대기(Explicit Wait) WebDrieverWait + expected_conditions 를 사용

In [ ]:
wait = WebDriverWait(driver, 10)

# 구(군) 선택 + 조회 + 엑셀 다운로드
def search_download_GuGun(guGun:str):
    # 1. 구(군) 지역 선택
    gu_elem = wait.until(
            EC.element_to_be_clickable(
                (By.ID, "SIGUNGU_NM0")
            )
    )  
    
    Select(gu_elem).select_by_visible_text(guGun)
 
    xpath = '//*[@id="templ_list0"]/div[8]/div/a'
    old_excel_btn = driver.find_element(By.XPATH, xpath)
    
    # 2.조회버튼 클릭
    search_btn =  wait.until(
            EC.element_to_be_clickable(
                (By.ID, "searRgSelect")
            )
    )
    search_btn.click()
 
    # 3. 기존 검색 결과가 교체될 때까지 대기
    wait.until(EC.staleness_of(old_excel_btn))
    
    new_excel_btn =  wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, xpath)
            )
    )
    new_excel_btn.click()
    # 동적 웹페이지에서 DOM 갱신 후 WebElement를 다시 찾는 원칙을 적용


# 시(도) 선택하여 해당 시(도)의 모둔 구(군) 다운로드

def search_download_sido(sido:str):
    # 1.시(도) 선택
    sido_elem = wait.until(
            EC.element_to_be_clickable((By.ID, 'SIDO_NM0')))
    Select(sido_elem).select_by_value(sido)
 
    # 2. 시(도) 선택 후 구(군) 목록이 생성될 때까지 대기
    wait.until(
        lambda driver:
            len(driver.find_elements(
                    By.CSS_SELECTOR, '#SIGUNGU_NM0 option')) > 1
    )
    
    # 3. 구(군) 목록 가져오기    
    guList = driver.find_element(By.ID, "SIGUNGU_NM0")
    guOpt = guList.find_elements(By.TAG_NAME, 'option')
    guNames = [opt.get_attribute('value') for opt in guOpt
                  if opt.get_attribute('value')]

    # 4. 선택한 시(도)의 모든 구(군) 조회 및 다운로드
    for guGun in guNames[1:]:
        search_download_GuGun(guGun)

| 기존코드 | 변경코드 | 의미 |
| :--- | :--- | :--- |
| `time.sleep(3)` | `element_to_be_clickable()` | 요소가 클릭 가능할 때까지 대기 |
| `time.sleep(3)` | `lambda driver:len(...)>1` | 구·군 `<option>` 이 실제 생성될 때까지 대기 |
| `send_keys()` | `Select().select_by_...()` | `<select>` 요소를 Selenium 표준방식 선택 |

In [61]:
driver.close()

- 다운로드 완료될 때 까지 대기

In [60]:
import os

# 파일 다운로드 완료 대기 함수
def download_complete(download_dir):
    files = os.listdir(download_dir)
    return not any(file.endswith('.crdownload') for file in files)

download_dir = 'C:/Users/acorn2/Downloads'
wait.until(
    lambda driver: download_complete(download_dir)
)

True

In [ ]:
from tqdm import tqdm

def search_download_sido(sido: str):
    # 1. 시(도) 선택
    sido_elem = wait.until(
        EC.element_to_be_clickable((By.ID, 'SIDO_NM0'))
    )
    Select(sido_elem).select_by_value(sido)
    
    # 2. 시(도) 선택 후 구(군) 목록이 생성될 때까지 대기
    wait.until(
        lambda driver:
            len(driver.find_elements(
                By.CSS_SELECTOR, '#SIGUNGU_NM0 option')) > 1
    )
    
    # 3. 구(군) 목록 가져오기
    guList = driver.find_element(By.ID, "SIGUNGU_NM0")
    guOpt = guList.find_elements(By.TAG_NAME, 'option')
    guNames = [opt.get_attribute('value') for opt in guOpt
               if opt.get_attribute('value')]
    
    # 4. 선택한 시(도)의 모든 구(군) 조회 및 다운로드
    for guGun in guNames[1:]:
        search_download_GuGun(guGun)

-----